# Retrieval Evaluation Query Generation

## 1. Introduction

### 1.1. Install and Import Libraries

In [8]:
# %pip uninstall -y torchvision
# %pip install --upgrade --force-reinstall --index-url https://download.pytorch.org/whl/cu128 torch==2.11.0+cu128 torchaudio==2.11.0+cu128
# %pip install --upgrade --force-reinstall "transformers==4.51.3" "huggingface_hub==0.30.2" "tokenizers==0.21.1" accelerate sentencepiece protobuf safetensors pandas tqdm
# %pip install -U bitsandbytes

In [7]:
# ============================================
# 1. Standard Library
# ============================================
import os
import json
from pathlib import Path
from importlib import reload

# ============================================
# 2. Third-party Libraries
# ============================================
import pandas as pd
import torch
from tqdm.auto import tqdm

# ============================================
# 3. Project Helpers
# ============================================
import src.retrieval_llms as retrieval_llm_helpers
reload(retrieval_llm_helpers)

from src.retrieval_llms import (
    QUERY_GENERATION_MODEL_ID,
    RELEVANCE_JUDGE_MODEL_ID,
    QueryGenerationLLM,
    RelevanceJudgeLLM,
    extract_first_json_object,
)

import src.retrieval_query_generation as query_generation_helpers
reload(query_generation_helpers)

from src.retrieval_query_generation import (
    dataframe_to_jsonl,
    deduplicate_qrels,
    filter_public_query_candidates,
    load_archive_chunks,
    load_generated_qrels,
    qrel_quality_summary,
    run_query_generation_llm,
    split_qrels_dev_test,
    stratified_sample_query_candidates,
    summarize_query_candidates,
)

In [3]:
# Dependency sanity check. If this fails after running the install cell,
# restart the notebook kernel and rerun imports before loading the LLMs.
import importlib.util
import transformers
import huggingface_hub
import torch

print(f"transformers: {transformers.__version__}")
print(f"huggingface_hub: {huggingface_hub.__version__}")
print(f"tokenizers installed: {importlib.util.find_spec('tokenizers') is not None}")
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA runtime: {torch.version.cuda}")
print(f"CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}")
print(f"torchvision installed: {importlib.util.find_spec('torchvision') is not None}")

transformers: 4.51.3
huggingface_hub: 0.30.2
tokenizers installed: True
torch: 2.11.0+cu128
CUDA available: True
CUDA runtime: 12.8
CUDA device: NVIDIA GeForce RTX 2060
torchvision installed: False


### 1.2. Initialize LLM Services

In [4]:
# Read the Hugging Face token from Colab userdata when available, then fall back to the environment.
try:
    from google.colab import userdata
    HUGGING_FACE_TOKEN = userdata.get("HUGGING_FACE_TOKEN")
except Exception:
    HUGGING_FACE_TOKEN = os.getenv("HUGGING_FACE_TOKEN")

LLM_LOAD_IN_4BIT = True
LLM_TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else "auto"

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

print(f"Query generation model: {QUERY_GENERATION_MODEL_ID}")
print(f"Relevance judge model: {RELEVANCE_JUDGE_MODEL_ID}")
print(f"Hugging Face token configured: {bool(HUGGING_FACE_TOKEN)}")

CUDA available: True
CUDA device: NVIDIA GeForce RTX 2060
Query generation model: Qwen/Qwen2.5-7B-Instruct
Relevance judge model: google/gemma-2-9b-it
Hugging Face token configured: True


Initialize the query-generation LLM. This model is used to generate realistic public-archive retrieval queries and qrels.

In [5]:
query_generation_llm = QueryGenerationLLM(
    token=HUGGING_FACE_TOKEN,
    torch_dtype=LLM_TORCH_DTYPE,
    load_in_4bit=LLM_LOAD_IN_4BIT,
    require_cuda=True,
    allow_cpu_offload=False,
)

print(query_generation_llm.device_summary())

W0523 22:31:38.790000 19564 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{'model_id': 'Qwen/Qwen2.5-7B-Instruct', 'device': 'cuda:0', 'hf_device_map': {'': 0}, 'devices': ['0'], 'has_cpu_or_disk_offload': False}


Verify the query-generation LLM returns parseable JSON.

In [14]:
llm_smoke_test_chunk = {
    "chunk_id": "smoke_test_chunk",
    "dataset": "smoke_test",
    "modality": "text",
    "title": "Archive excerpt",
    "access_level": "public",
    "sensitivity_level": "low",
    "masked_text": (
        "A city council report says the Riverside Public Library opened a digital archive "
        "in 2024 to preserve oral histories, photographs, and meeting records from local residents."
    ),
}

llm_smoke_test_response = query_generation_llm.generate_query_record(
    llm_smoke_test_chunk,
    max_new_tokens=160,
)
print(llm_smoke_test_response)

llm_smoke_test_record = extract_first_json_object(llm_smoke_test_response)
display(pd.DataFrame([llm_smoke_test_record]))

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{
  "reject": false,
  "query": "When did the Riverside Public Library open its digital archive?",
  "expected_relevant_information": "The year 2024.",
  "reference_answer": "2024",
  "query_type": "factual_lookup",
  "difficulty": "easy",
  "grounding_evidence": "2024 to preserve oral histories, photographs, and meeting records",
  "quality_notes": "This query requires a specific factual detail that can be directly answered from the provided text."
}


,reject,query,expected_relevant_information,reference_answer,query_type,difficulty,grounding_evidence,quality_notes
0,False,When did the Riverside Public Library open its...,The year 2024.,2024,factual_lookup,easy,"2024 to preserve oral histories, photographs, ...",This query requires a specific factual detail ...


## 2. Generate Retrieval Evaluation Queries

### 2.1. Load archive chunks

In [ ]:
RETRIEVAL_QUERY_EXPORT_DIR = Path("data/evaluation/retrieval_queries")
RETRIEVAL_QUERY_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_CHUNKS_CSV = Path("data/graph_kb_exports/step_01_archive_skeleton/chunks.csv")
GENERATED_QRELS_JSONL = RETRIEVAL_QUERY_EXPORT_DIR / "retrieval_qrels_generated.jsonl"
DEV_QRELS_JSONL = RETRIEVAL_QUERY_EXPORT_DIR / "retrieval_qrels_dev.jsonl"
TEST_QRELS_JSONL = RETRIEVAL_QUERY_EXPORT_DIR / "retrieval_qrels_test.jsonl"
QUERY_GENERATION_PROGRESS_JSONL = RETRIEVAL_QUERY_EXPORT_DIR / "retrieval_query_generation_progress.jsonl"
QUERY_GENERATION_REJECTS_JSONL = RETRIEVAL_QUERY_EXPORT_DIR / "retrieval_query_generation_rejects.jsonl"
QUERY_CANDIDATE_QUEUE_JSONL = RETRIEVAL_QUERY_EXPORT_DIR / "retrieval_query_candidate_queue.jsonl"

archive_chunks_df = load_archive_chunks(ARCHIVE_CHUNKS_CSV, verbose=True)
print(f"Loaded archive chunks: {len(archive_chunks_df):,}")
display(archive_chunks_df.head(3))

### 2.2. Candidate filtering and inspection

In [17]:
QUERY_MIN_TOKENS = 45
QUERY_MIN_UNIQUE_TOKENS = 20
QUERY_MIN_ALPHA_RATIO = 0.45
QUERY_MAX_PUNCTUATION_NOISE_RATIO = 0.35

candidate_summary_tables = summarize_query_candidates(archive_chunks_df)
for table_name, table_df in candidate_summary_tables.items():
    print(table_name)
    display(table_df.head(40))

public_query_candidates_df = filter_public_query_candidates(
    archive_chunks_df,
    min_tokens=QUERY_MIN_TOKENS,
    min_unique_tokens=QUERY_MIN_UNIQUE_TOKENS,
    min_alpha_ratio=QUERY_MIN_ALPHA_RATIO,
    max_punctuation_noise_ratio=QUERY_MAX_PUNCTUATION_NOISE_RATIO,
    verbose=True,
)

print(f"Public query candidates after filtering: {len(public_query_candidates_df):,}")
display(
    public_query_candidates_df.groupby(["dataset", "modality", "length_bucket"], dropna=False)
    .size()
    .reset_index(name="candidate_chunks")
    .sort_values(["dataset", "modality", "length_bucket"])
)
display(public_query_candidates_df[["chunk_id", "dataset", "modality", "title", "text_token_count", "unique_token_count", "length_bucket", "masked_text"]].head(10))

access_level


,access_level,chunks
0,public,297016


quality


,access_level,dataset,modality,query_candidate_quality,chunks
0,public,cnn_dailymail,article,ocr_noise,1
1,public,cnn_dailymail,article,too_short,34
2,public,cnn_dailymail,article,usable,153292
3,public,docvqa,ocr_document,ocr_noise,823
4,public,docvqa,ocr_document,too_short,38
5,public,docvqa,ocr_document,usable,1498
6,public,mediasum,transcript,too_short,17
7,public,mediasum,transcript,usable,134800
8,public,msr_vtt,video,ocr_noise,4
9,public,msr_vtt,video,too_short,6509


length


,access_level,dataset,modality,length_bucket,chunks
0,public,cnn_dailymail,article,long,144149
1,public,cnn_dailymail,article,medium,8992
2,public,cnn_dailymail,article,short,183
3,public,cnn_dailymail,article,very_long,3
4,public,docvqa,ocr_document,long,1696
5,public,docvqa,ocr_document,medium,480
6,public,docvqa,ocr_document,short,168
7,public,docvqa,ocr_document,very_long,15
8,public,mediasum,transcript,long,127069
9,public,mediasum,transcript,medium,7534


Public query candidates after filtering: 289,612


,dataset,modality,length_bucket,candidate_chunks
0,cnn_dailymail,article,long,144149
1,cnn_dailymail,article,medium,8992
2,cnn_dailymail,article,short,164
3,cnn_dailymail,article,very_long,2
4,docvqa,ocr_document,long,1183
5,docvqa,ocr_document,medium,267
6,docvqa,ocr_document,short,53
7,mediasum,transcript,long,127069
8,mediasum,transcript,medium,7534
9,mediasum,transcript,short,197


,chunk_id,dataset,modality,title,text_token_count,unique_token_count,length_bucket,masked_text
0,0000066fbb9ca4b00702658c,cnn_dailymail,article,CNN/DailyMail Article 656dd639044d,137,101,medium,"Regardless of the trek you choose, the dry sea..."
1,00009fe0db7962bf34af45b5,mediasum,transcript,Congress Has 16 Days To Avoid Another Shutdown,229,138,long,"SUSAN DAVIS, BYLINE: Well, the president - we ..."
2,0000a5e7aaf3cfa873de655e,cnn_dailymail,article,CNN/DailyMail Article ed51901ec69b,332,185,long,"And in Washington, President Obama said in a s..."
3,0000bdefb1ab4fe16016df97,mediasum,transcript,Removing Religion from Holidays a Tall Order,294,148,long,TOVIA SMITH: Epstein dreams of a day when huma...
4,00011ca6fa4f615d994679e3,cnn_dailymail,article,CNN/DailyMail Article 59f5ed65da9e,264,166,long,"ON PUGET SOUND, Washington (CNN) -- When comme..."
5,000121a4c655c7027b5d5579,mediasum,transcript,Understanding CSRs In The Health Care Debate,327,169,long,"JULIE MIX MCPEAK: Well, certainly the discussi..."
6,000140d6bf72cd431ec4a398,cnn_dailymail,article,CNN/DailyMail Article 7d6190ea8690,348,188,long,"Once Liv returned, reformed assassin and perso..."
7,00017c18aa880000e75a80f1,cnn_dailymail,article,CNN/DailyMail Article fbcf77d4da17,167,125,medium,"Speaking to reporters after the match, Barca c..."
8,0001d34c5ffcd5b4f9b60923,mediasum,transcript,Mining the Disabled List for Insight Into Base...,355,139,long,"IRA FLATOW, host: Correct. They happened, like..."
9,0001e3a6f225dffb201f5456,cnn_dailymail,article,CNN/DailyMail Article 60dfbd20c88d,274,149,long,Pilots need to pay attention even when they ar...


### 2.3. Stratified sampling

In [18]:
QUERY_GENERATION_RANDOM_STATE = 42
TARGET_GENERATED_QRELS = 300  # Dry run. Increase to 100-300 for the main evaluation set.
QUERY_CANDIDATE_RESERVE_MULTIPLIER = 5

# The reserve queue is intentional: OCR/noisy chunks may pass heuristics but be rejected by the LLM.
# The generation step consumes this queue until enough accepted qrels are produced.
query_candidate_queue_df = stratified_sample_query_candidates(
    public_query_candidates_df,
    target_queries=TARGET_GENERATED_QRELS,
    reserve_multiplier=QUERY_CANDIDATE_RESERVE_MULTIPLIER,
    strata_columns=["dataset", "modality", "length_bucket"],
    random_state=QUERY_GENERATION_RANDOM_STATE,
    verbose=True,
)

dataframe_to_jsonl(query_candidate_queue_df, QUERY_CANDIDATE_QUEUE_JSONL)
print(f"Candidate queue size: {len(query_candidate_queue_df):,}")
print(f"Candidate queue exported to: {QUERY_CANDIDATE_QUEUE_JSONL}")
display(
    query_candidate_queue_df.groupby(["dataset", "modality", "length_bucket"], dropna=False)
    .size()
    .reset_index(name="queued_chunks")
    .sort_values(["dataset", "modality", "length_bucket"])
)
display(query_candidate_queue_df[["candidate_rank", "chunk_id", "dataset", "modality", "length_bucket", "title", "masked_text"]].head(10))

Candidate queue size: 1,500
Candidate queue exported to: data\evaluation\retrieval_queries\retrieval_query_candidate_queue.jsonl


,dataset,modality,length_bucket,queued_chunks
0,cnn_dailymail,article,long,357
1,cnn_dailymail,article,medium,136
2,cnn_dailymail,article,short,125
3,cnn_dailymail,article,very_long,2
4,docvqa,ocr_document,long,127
5,docvqa,ocr_document,medium,125
6,docvqa,ocr_document,short,53
7,mediasum,transcript,long,312
8,mediasum,transcript,medium,136
9,mediasum,transcript,short,125


,candidate_rank,chunk_id,dataset,modality,length_bucket,title,masked_text
0,1,542cfeb6cf313040544446f5,cnn_dailymail,article,long,CNN/DailyMail Article 8a491aaa331d,(Departures.com) -- St. Andrews and the surrou...
1,2,52625a43e49d22bb668a10fe,cnn_dailymail,article,long,CNN/DailyMail Article 1af7169c8e46,"ATLANTA, Georgia (CNN) -- You don't have to te..."
2,3,79a5ede1b9ee1776377a687d,docvqa,ocr_document,long,DocVQA Page glxm0052::page-3,"OCR Line 28 [538,543,834,565]: R J REYNOLDS TO..."
3,4,fa0d9120854dfa448f4c2dbe,cnn_dailymail,article,medium,CNN/DailyMail Article 5c1c92a46e53,"National Geospatial-Intelligence Agency, which..."
4,5,67c9fc986896785101803eba,cnn_dailymail,article,long,CNN/DailyMail Article 5a7ddd513889,Department of Labor agency and said it would c...
5,6,15d60b153bccfd264b3356f2,docvqa,ocr_document,long,DocVQA Page yrgj0223::page-33,"OCR Line 80 [1082,1621,1616,1649]: share in in..."
6,7,06261adb87e21458257ef7d8,cnn_dailymail,article,long,CNN/DailyMail Article c605e2d82d89,"Kunz, who runs an independent car rental compa..."
7,8,92507bb09175d2ddc7f40a93,cnn_dailymail,article,long,CNN/DailyMail Article 96f567d274f4,But while many Twitter users with a dose of ce...
8,9,ef8159ada2ce69312e7f48af,mediasum,transcript,long,Former Cabinet Members On Being Part Of The Pr...,"MADELEINE ALBRIGHT: Well, I think - as a I sai..."
9,10,2d4ee7be5cce7328923f41b0,cnn_dailymail,article,medium,CNN/DailyMail Article 1fb172f727a7,Saleh's relatives still hold key military and ...


### 2.4. Query generation prompt

In [19]:
QUERY_GENERATION_PROMPT_VERSION = "v1"
QUERY_GENERATION_MAX_NEW_TOKENS = 768
QUERY_GENERATION_TEMPERATURE = 0.2
QUERY_GENERATION_TOP_P = 0.9
QUERY_GENERATION_DO_SAMPLE = True

print("Prompt lives in QueryGenerationLLM.generate_query_record.")
print("Key constraints: public-only, masked_text only, self-contained expected information, no source-framing terms, reject noisy chunks.")
print(f"Prompt version: {QUERY_GENERATION_PROMPT_VERSION}")
display(query_candidate_queue_df[["chunk_id", "dataset", "modality", "title", "masked_text"]].head(1))

Prompt lives in QueryGenerationLLM.generate_query_record.
Key constraints: public-only, masked_text only, self-contained expected information, no source-framing terms, reject noisy chunks.
Prompt version: v1


,chunk_id,dataset,modality,title,masked_text
0,542cfeb6cf313040544446f5,cnn_dailymail,article,CNN/DailyMail Article 8a491aaa331d,(Departures.com) -- St. Andrews and the surrou...


### 2.5. Run query-generation LLM

In [12]:
RUN_QUERY_GENERATION_LLM = True

if not RUN_QUERY_GENERATION_LLM:
    print("Query generation is disabled. Set RUN_QUERY_GENERATION_LLM = True to start/resume generation.")
else:
    if "query_generation_llm" not in globals():
        raise RuntimeError("Initialize query_generation_llm in section 1.2 before running generation.")

    query_generation_summary = run_query_generation_llm(
        query_candidate_queue_df,
        query_generation_llm,
        output_jsonl=GENERATED_QRELS_JSONL,
        progress_jsonl=QUERY_GENERATION_PROGRESS_JSONL,
        rejects_jsonl=QUERY_GENERATION_REJECTS_JSONL,
        target_qrels=TARGET_GENERATED_QRELS,
        prompt_version=QUERY_GENERATION_PROMPT_VERSION,
        max_new_tokens=QUERY_GENERATION_MAX_NEW_TOKENS,
        temperature=QUERY_GENERATION_TEMPERATURE,
        top_p=QUERY_GENERATION_TOP_P,
        do_sample=QUERY_GENERATION_DO_SAMPLE,
        verbose=True,
    )
    display(pd.DataFrame([{k: str(v) if isinstance(v, Path) else v for k, v in query_generation_summary.items()}]))

NameError: name 'GENERATED_QRELS_JSONL' is not defined

### 2.6. Validate and deduplicate generated qrels

In [ ]:
generated_qrels_df = load_generated_qrels(GENERATED_QRELS_JSONL)
valid_generated_qrels_df = generated_qrels_df[generated_qrels_df["is_valid"].fillna(True)].copy() if not generated_qrels_df.empty else generated_qrels_df.copy()
deduped_generated_qrels_df = deduplicate_qrels(valid_generated_qrels_df)

print(f"Generated qrels: {len(generated_qrels_df):,}")
print(f"Valid generated qrels: {len(valid_generated_qrels_df):,}")
print(f"Deduplicated qrels: {len(deduped_generated_qrels_df):,}")
display(deduped_generated_qrels_df.head(10))

if QUERY_GENERATION_REJECTS_JSONL.exists():
    rejected_generation_df = pd.DataFrame(query_generation_helpers.read_jsonl(QUERY_GENERATION_REJECTS_JSONL))
    print(f"Rejected/error records: {len(rejected_generation_df):,}")
    display(rejected_generation_df.tail(10))

### 2.7. Export dev/test qrels

In [ ]:
DEV_QREL_FRACTION = 0.3

dev_qrels_df, test_qrels_df = split_qrels_dev_test(
    deduped_generated_qrels_df,
    dev_fraction=DEV_QREL_FRACTION,
    random_state=QUERY_GENERATION_RANDOM_STATE,
    stratify_columns=["dataset", "modality", "query_type"],
)

dataframe_to_jsonl(dev_qrels_df, DEV_QRELS_JSONL)
dataframe_to_jsonl(test_qrels_df, TEST_QRELS_JSONL)

print(f"Dev qrels: {len(dev_qrels_df):,} -> {DEV_QRELS_JSONL}")
print(f"Test qrels: {len(test_qrels_df):,} -> {TEST_QRELS_JSONL}")

### 2.8. Preview query set quality

In [ ]:
qrel_quality_tables = qrel_quality_summary(deduped_generated_qrels_df)
for table_name, table_df in qrel_quality_tables.items():
    print(table_name)
    display(table_df)

preview_columns = [
    "query_id",
    "query",
    "expected_relevant_information",
    "reference_answer",
    "query_type",
    "difficulty",
    "dataset",
    "modality",
    "source_chunk_id",
]
display(deduped_generated_qrels_df[[col for col in preview_columns if col in deduped_generated_qrels_df.columns]].head(20))